In [1]:
# pip install youtube-transcript-api

In [1]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
load_dotenv()
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled


# Step 1 I ndexing (Document Ingestion)

In [2]:
video_id = "7ARBJQn6QkM"
try:
    transcript_list = YouTubeTranscriptApi().fetch(video_id)

    transcript = " ".join(chunk.text for chunk in transcript_list)

    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video")

At some point, you have to believe something. 
We've reinvented computing as we know it. What is the vision for what you see coming next? We 
asked ourselves, if it can do this, how far can it go? How do we get from the robots that 
we have now to the future world that you see? Cleo, everything that moves will be 
robotic someday and it will be soon. We invested tens of billions of dollars before 
it really happened. No that's very good, you did some research! But the big breakthrough 
I would say is when we... That's Jensen Huang, and whether you know it or not
his decisions are shaping your future. He's the CEO of NVIDIA, the company that skyrocketed over the past few
years to become one of the most valuable companies in the world because they led a fundamental shift 
in how computers work unleashing this current explosion of what's possible with technology. 
"NVIDIA's done it again!" We found ourselves being one of the most important technology companies in 
the world and potentiall

# text splitters

In [3]:
splitter =RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.create_documents([transcript])
len(chunks)
chunks[10]

Document(metadata={}, page_content="oh my god it's revolutionizing all of these other industries as well, it's beginning to change\xa0\nhow we see what's possible with computers and my understanding is that in the early 2000s you\xa0\nsee this and you realize that actually doing that is a little bit difficult because what that\xa0\nresearcher had to do is he had to sort of trick the GPUs into thinking that his problem was a\xa0\ngraphics problem. That's exactly right, no that's very good, you did some research. So you create\xa0\na way to make that a lot easier. That's right Specifically it's a platform called CUDA which\xa0\nlets programmers tell the GPU what to do using programming languages that they already know like\xa0\nC and that's a big deal because it gives way more people easier access to all of this computing\xa0\npower. Could you explain what the vision was that led you to create CUDA? Partly researchers\xa0\ndiscovering it, partly internal inspiration and and partly solvin

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
vector_store=FAISS.from_documents(chunks,embeddings)

In [11]:
vector_store.index_to_docstore_id

{0: 'a70416c0-691c-479e-a975-d78b667f71c7',
 1: '4cd14830-02c8-4d4e-bc2e-6dcfd1d9c265',
 2: '98c3de4e-75a2-45df-9653-72292dbf4fb4',
 3: 'a00733f1-494f-4809-84c0-4ddce0dfc822',
 4: '226bd106-8fda-4a70-85b5-59fdd7773e1b',
 5: 'f00fa8ae-ce48-43e9-a5ee-c32afdecdb8f',
 6: 'cc4d5f48-bfd7-4b18-952c-300474724ee1',
 7: '3081f94c-f316-454d-a228-cac7c43a83de',
 8: '84ef373a-6ccc-4a7d-872f-a3b337029b06',
 9: '50f47e8c-7a4e-4fac-8ac1-17a7a219b2c1',
 10: 'c29327ab-e9ca-4e32-b330-eac42c92cf25',
 11: 'f4d43254-de0d-4eba-9642-ad23c52b3c08',
 12: '11c54e73-d53d-4d11-8e58-21f68480708b',
 13: 'e5ba1d80-f5f7-46bd-8f96-8d0213a1ef5e',
 14: '0dd649ca-7b7d-4295-88c4-2abe62f5e488',
 15: '261ea75d-6abc-4bc1-a816-7730fa82c8a7',
 16: '8ac295f3-10e7-4408-aa25-0d8c8513ab0c',
 17: '91f1930b-2190-4ece-89b6-b7804ce6635a',
 18: 'bb33b765-71b8-4c35-9f7f-6fd4bd46d52a',
 19: '601dbce6-09c0-4fef-a69f-b0adf13174e9',
 20: 'ee6bed6e-0e1e-4fb3-ae0a-f4c836028b4c',
 21: '94c92e7c-1a12-4ed0-a810-ef215f9f3e30',
 22: 'd4cbb221-4682-

In [13]:

vector_store.get_by_ids(['a2427d9b-7bb2-498b-bc0e-a4bc6a5bf39f'])

[Document(id='a2427d9b-7bb2-498b-bc0e-a4bc6a5bf39f', metadata={}, page_content="place. That if we wanted to drive we can drive but otherwise you know take a nap or enjoy\xa0\nyour car like it's a home theater of yours, you know read from work to home and at that\xa0\npoint you're hoping that you live far away and so you could be in a car for longer.\xa0\nAnd you look back and you realize that there's this company almost at\xa0\nthe epicenter of all of that and happens to be the company that\nyou grew up playing games\xa0with. I hope for that to be\nwhat the next generation learn. Thank you so much\xa0for your time.\nI enjoyed it, thank you! I'm glad!")]

In [17]:
retriever=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [18]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001C62D7DC200>, search_kwargs={'k': 4})

In [19]:
retriever.invoke('what is the impact of AI')

[Document(id='832b03e0-c750-4023-8c9d-1db20c3a47c8', metadata={}, page_content="somebody. You know for example self-driving car that wants to drive nicely and drive properly\xa0\nand just somehow the sensor broke down or it didn't detect something. Or you know made it\xa0\ntoo aggressive turn or whatever it is. It did it poorly. It did it wrongly. And so that's\na whole bunch of engineering that has to be done to to make sure that AI safety is upheld\xa0\nby making sure that the product functions properly. And then and then lastly you know whatever what\xa0\nhappens if the system, the AI wants to do a good job but the system failed. Meaning the AI wanted\xa0\nto stop something from happening and it turned out just when it wanted to do\xa0\nit, the machine broke down. And so this is no different than a flight computer inside\xa0\na plane having three versions of them and then so there's triple redundancy inside the\xa0\nsystem inside autopilots and then you have two pilots and then you 

In [22]:
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace

In [23]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.3-70B-Instruct",
    task="text-generation",
    temperature=0
)

llm = ChatHuggingFace(llm=llm)

In [25]:
prompt=PromptTemplate(
    template="""
    You are a helpful AI assistant. Answer ONLY from the following provided transcript context.
      If the context is insufficient, just say you don't know. 
            
      {context}
       Question : {question} 
    input_variables = ['context'. ['question']]
    """
    
)

In [26]:
question="What are we going to have in the next 10 tears?"
retrieved_docs=retriever.invoke(question)
retrieved_docs

[Document(id='11bf5ee4-2101-4a57-80d6-27cd02ed61a8', metadata={}, page_content="it opened up this universe of opportunities and universe of problems that we can go solve. And\xa0\nthat gets us quite excited. It feels like we are on the cusp of this truly enormous change.\xa0\nWhen I think about the next 10 years, unlike the last 10 years, I know we've gone through a lot of\xa0\nchange already but I don't think I can predict anymore how I will be using the technology that is\xa0\ncurrently being developed. That's exactly right. I think the last 10, the reason why you feel that way\xa0\nis, the last 10 years was really about the science of AI. The next 10 years we're going to have plenty\xa0\nof science of AI but the next 10 years is going to be the application science of AI. The fundamental\xa0\nscience versus the application science. And so the the applied research, the application side of AI\xa0\nnow becomes: How can I apply AI to digital biology? How can I apply AI to climate technol

In [27]:

context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'it opened up this universe of opportunities and universe of problems that we can go solve. And\xa0\nthat gets us quite excited. It feels like we are on the cusp of this truly enormous change.\xa0\nWhen I think about the next 10 years, unlike the last 10 years, I know we\'ve gone through a lot of\xa0\nchange already but I don\'t think I can predict anymore how I will be using the technology that is\xa0\ncurrently being developed. That\'s exactly right. I think the last 10, the reason why you feel that way\xa0\nis, the last 10 years was really about the science of AI. The next 10 years we\'re going to have plenty\xa0\nof science of AI but the next 10 years is going to be the application science of AI. The fundamental\xa0\nscience versus the application science. And so the the applied research, the application side of AI\xa0\nnow becomes: How can I apply AI to digital biology? How can I apply AI to climate technology? How can\n\ntalked about CUDA, we\'ve talked about bets you\'ve made in

In [28]:

final_prompt = prompt.invoke({"context":context_text, "question":question})

In [29]:

final_prompt

StringPromptValue(text='\n    You are a helpful AI assistant. Answer ONLY from the following provided transcript context.\n      If the context is insufficient, just say you don\'t know. \n\n      it opened up this universe of opportunities and universe of problems that we can go solve. And\xa0\nthat gets us quite excited. It feels like we are on the cusp of this truly enormous change.\xa0\nWhen I think about the next 10 years, unlike the last 10 years, I know we\'ve gone through a lot of\xa0\nchange already but I don\'t think I can predict anymore how I will be using the technology that is\xa0\ncurrently being developed. That\'s exactly right. I think the last 10, the reason why you feel that way\xa0\nis, the last 10 years was really about the science of AI. The next 10 years we\'re going to have plenty\xa0\nof science of AI but the next 10 years is going to be the application science of AI. The fundamental\xa0\nscience versus the application science. And so the the applied research, 

In [31]:
answer = llm.invoke(final_prompt)
print(answer.content)

In the next 10 years, we're going to have plenty of science of AI, but it's going to be the application science of AI. The applied research and application side of AI will focus on areas such as digital biology, climate technology, and more.


# buildina a chain 

In [32]:

from langchain_core.runnables import RunnableParallel, RunnableSequence, RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text
parallel_chain =RunnableParallel({
    'context' : retriever | RunnableLambda(format_docs),
    'question' : RunnablePassthrough()
})

In [33]:
parallel_chain.invoke("What are we going to have in the next 10 years?")

{'context': "it opened up this universe of opportunities and universe of problems that we can go solve. And\xa0\nthat gets us quite excited. It feels like we are on the cusp of this truly enormous change.\xa0\nWhen I think about the next 10 years, unlike the last 10 years, I know we've gone through a lot of\xa0\nchange already but I don't think I can predict anymore how I will be using the technology that is\xa0\ncurrently being developed. That's exactly right. I think the last 10, the reason why you feel that way\xa0\nis, the last 10 years was really about the science of AI. The next 10 years we're going to have plenty\xa0\nof science of AI but the next 10 years is going to be the application science of AI. The fundamental\xa0\nscience versus the application science. And so the the applied research, the application side of AI\xa0\nnow becomes: How can I apply AI to digital biology? How can I apply AI to climate technology? How can\n\ntalked about CUDA, we've talked about bets you've m

In [34]:
parser = StrOutputParser()
final_chain=parallel_chain | prompt | llm |parser

In [35]:
result=final_chain.invoke("what are we going to have in the next 10 years")

In [37]:
print(result)

In the next 10 years, we can expect significant advancements in the application of AI, including:

1. The fusion of Omniverse and Cosmos, creating a new type of generative world generation system.
2. Major developments in human robotics, with robots learning much faster through digital simulation.
3. Transformations in the field of digital biology and life sciences.
4. A revolution in material sciences.
5. Widespread use of robots to perform dangerous and mundane tasks.
6. Self-driving cars becoming the norm, allowing people to use their cars like home theaters.
7. Everything that moves becoming robotic, including lawn mowers and cars, with humanoid robots emerging as a possibility.

These advancements will be driven by the shift from the science of AI to the application science of AI, leading to profound changes in various fields and aspects of life.
